In [1]:
"""
Sankey Plot: Design Principle → Descriptive Codes → Student Reactions
Data source: db-q4.csv
Requirements: pip install pandas plotly kaleido
"""

import pandas as pd
import plotly.graph_objects as go

# int_var2 = "Descriptive Codes"

int_var = "Descriptive Codes"
# int_var = "Exact question"


# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv("db-4-5.csv")
df.columns = df.columns.str.strip()
df = df.dropna(subset=["Design Principle", int_var, "Students reactions"])

# ── 2. Build node list (order: Design Principles → Descriptive Codes → Reactions)
design_principles = sorted(df["Design Principle"].unique().tolist())
descriptive_codes = sorted(df[int_var].unique().tolist())
student_reactions = sorted(df["Students reactions"].unique().tolist())

all_nodes = design_principles + descriptive_codes + student_reactions

node_index = {name: i for i, name in enumerate(all_nodes)}

dp_count  = len(design_principles)
dc_count  = len(descriptive_codes)
sr_count  = len(student_reactions)

In [2]:
## OFFICIAL

# ── 2b. Select 20 codes: top-10 most popular + 10 most unique ─────────────────
dc_counts = df[int_var].value_counts()
top10_popular = dc_counts.nlargest(10).index.tolist()
remaining_counts = dc_counts.drop(top10_popular)
top10_unique = remaining_counts.nsmallest(10).index.tolist()
selected_codes = set(top10_popular + top10_unique)

# ── Visibility config ─────────────────────────────────────────────────────────
SHOW_MIN    = 2
ALWAYS_SHOW = ["team", "teamwork", "collaborat"]
ALWAYS_HIDE = ["system structure easy to navigate"]

def matches(label, keywords):
    return any(kw in label.lower() for kw in keywords)

# ── 2c. Rebuild node lists excluding hidden nodes ─────────────────────────────
vis_dp = [n for n in design_principles if not matches(n, ALWAYS_HIDE)]
vis_dc = [n for n in descriptive_codes  if not matches(n, ALWAYS_HIDE)]
vis_sr = [n for n in student_reactions  if not matches(n, ALWAYS_HIDE)]

all_nodes_vis  = vis_dp + vis_dc + vis_sr
node_index_vis = {name: i for i, name in enumerate(all_nodes_vis)}

vdp, vdc, vsr = len(vis_dp), len(vis_dc), len(vis_sr)

# ── 3. Build links ────────────────────────────────────────────────────────────
layer1 = df.groupby(["Design Principle", int_var]).size().reset_index(name="value")
layer2 = df.groupby([int_var, "Students reactions"]).size().reset_index(name="value")

sources, targets, values = [], [], []

for _, row in layer1.iterrows():
    a, b = row["Design Principle"], row[int_var]
    if a not in node_index_vis or b not in node_index_vis:
        continue
    sources.append(node_index_vis[a])
    targets.append(node_index_vis[b])
    values.append(row["value"])

for _, row in layer2.iterrows():
    a, b = row[int_var], row["Students reactions"]
    if a not in node_index_vis or b not in node_index_vis:
        continue
    sources.append(node_index_vis[a])
    targets.append(node_index_vis[b])
    values.append(row["value"])

# ── 4. Colour palettes ────────────────────────────────────────────────────────
# Design Principles: cool/muted tones
dp_color_list = ["#4C72B0", "#8172B2", "#55A868", "#5A8F7B", "#DD8452"]
dp_colors = dp_color_list[:vdp]
dp_color_map = {dp: c for dp, c in zip(vis_dp, dp_colors)}

# Descriptive Codes: inherit dominant Design Principle color
def dominant_dp_color(dc_name):
    sub = df[df[int_var] == dc_name]["Design Principle"].value_counts()
    if sub.empty:
        return "#CCCCCC"
    return dp_color_map.get(sub.index[0], "#CCCCCC")

dc_colors = [dominant_dp_color(dc) for dc in vis_dc]

# Student Reactions: warm/earthy palette — clearly distinct from DP hue families
sr_palette = [
    "#C0392B",  # crimson red
    "#E91E63",  # hot pink
    "#F1C40F",  # gold
    "#03A9F4",  # sky blue (more cyan than DP blue)
    "#795548",  # brown
    "#607D8B",  # slate blue-grey
    "#00897B",  # teal-green
    "#FF7043",  # deep orange
]
sr_colors = sr_palette[:vsr]

node_colors = dp_colors + dc_colors + sr_colors

# ── 4b. Labels ────────────────────────────────────────────────────────────────
dc_labels  = [code if code in selected_codes else "" for code in vis_dc]
all_labels = vis_dp + dc_labels + vis_sr

# ── 4c. Link colours ─────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, alpha=0.35):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(node_colors[s])
    if (v > SHOW_MIN or matches(all_labels[s], ALWAYS_SHOW) or matches(all_labels[t], ALWAYS_SHOW))
    else "rgba(0,0,0,0)"
    for s, t, v in zip(sources, targets, values)
]

# ── 4d. Node x/y positions ───────────────────────────────────────────────────
x_dp, x_dc, x_sr = 0.01, 2/6, 0.99

def col_y(n):
    if n == 1:
        return [0.5]
    return [0.02 + i * 0.96 / (n - 1) for i in range(n)]

def flow_proportional_y(items, data, col, pad_norm):
    """Position node centers proportionally to flow so bars never clip neighbors."""
    flows = [data[data[col] == item].shape[0] for item in items]
    total = sum(flows)
    n = len(items)
    if n <= 1:
        return [0.5]
    usable = 1.0 - pad_norm * (n - 1)
    centers, cum = [], 0.0
    for f in flows:
        h = f / total * usable
        centers.append(cum + h / 2)
        cum += h + pad_norm
    return centers

node_x = [x_dp] * vdp + [x_dc] * vdc + [x_sr] * vsr
node_y = (flow_proportional_y(vis_dp, df, "Design Principle",   pad_norm=8/450)
        + col_y(vdc)
        + flow_proportional_y(vis_sr, df, "Students reactions", pad_norm=8/450))

# ── 5. Build figure ───────────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="fixed",
    node=dict(
        pad=8,
        thickness=14,
        line=dict(color="white", width=0.5),
        label=all_labels,
        color=node_colors,
        x=node_x,
        y=node_y,
        hovertemplate="<b>%{label}</b><br>Total flow: %{value}<extra></extra>",
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b>"
            "<br>Count: %{value}<extra></extra>"
        ),
    ),
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Design Principle → Descriptive Codes → Student Reactions</b>"
            "<br><sup>Each band width is proportional to the number of observations</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=24),
    ),
    font=dict(family="Arial", size=16, color="#333333"),
    paper_bgcolor="white",
    height=450,
    width=1400,
    margin=dict(l=20, r=20, t=80, b=10),
)

# ── 6. Layer label annotations ────────────────────────────────────────────────
for x_pos, label in zip([0.01, 1/6, 0.99], ["Design Principles", int_var, "Student Reactions"]):
    fig.add_annotation(
        x=x_pos, y=1.06,
        xref="paper", yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(size=20, color="#444"),
        align="center",
    )

# ── 7. Save outputs ───────────────────────────────────────────────────────────
fig.write_html("sankey_output.html")
print("✅  Saved: sankey_output.html  (interactive)")

try:
    fig.write_image("sankey_output.png", scale=2)
    print("✅  Saved: sankey_output.png   (static, 2× resolution)")
except Exception as e:
    print(f"⚠️  PNG export skipped ({e}). Install kaleido: pip install kaleido")

fig.show()


✅  Saved: sankey_output.html  (interactive)
⚠️  PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). Install kaleido: pip install kaleido


In [ ]:
## SECOND FIGURE

# ── Visibility config ─────────────────────────────────────────────────────────
SHOW_MIN    = 2
ALWAYS_SHOW = ["team", "teamwork", "collaborat"]
ALWAYS_HIDE = ["system structure easy to navigate"]

def matches(label, keywords):
    return any(kw in label.lower() for kw in keywords)

def hex_to_rgba(hex_color, alpha=0.35):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def col_y(n):
    if n == 1: return [0.5]
    return [0.02 + i * 0.96 / (n - 1) for i in range(n)]

def flow_proportional_y(items, data, col, pad_norm=8/600):
    """Space node centers by cumulative flow so heights are proportional to counts."""
    flows = [data[data[col] == item].shape[0] for item in items]
    total = sum(flows)
    n = len(items)
    if n <= 1:
        return [0.5]
    usable = 1.0 - pad_norm * (n - 1)
    centers, cum = [], 0.0
    for f in flows:
        h = f / total * usable
        centers.append(cum + h / 2)
        cum += h + pad_norm
    return centers

def wrap50(text, max_chars=50):
    words = text.split()
    lines, cur = [], ""
    for w in words:
        if len(cur) + len(w) + 1 <= max_chars:
            cur = (cur + " " + w).strip()
        else:
            if cur: lines.append(cur)
            cur = w
    if cur: lines.append(cur)
    return "<br>".join(lines)

# ── Node lists ────────────────────────────────────────────────────────────────
vis_dp  = [n for n in design_principles if not matches(n, ALWAYS_HIDE)]
vis_dc  = [n for n in descriptive_codes  if not matches(n, ALWAYS_HIDE)]
vis_sr  = [n for n in student_reactions  if not matches(n, ALWAYS_HIDE)]
hide_sr = [n for n in student_reactions  if     matches(n, ALWAYS_HIDE)]
n_dp, n_dc, n_sr = len(vis_dp), len(vis_dc), len(vis_sr)

dc_y = flow_proportional_y(vis_dc, df, int_var)

# ── Colour palettes ───────────────────────────────────────────────────────────
dp_color_list = ["#4C72B0", "#8172B2", "#55A868", "#5A8F7B", "#DD8452"]
dp_colors     = dp_color_list[:n_dp]
dp_color_map  = {dp: c for dp, c in zip(vis_dp, dp_colors)}

def dominant_dp_color(dc_name):
    sub = df[df[int_var] == dc_name]["Design Principle"].value_counts()
    return dp_color_map.get(sub.index[0], "#CCCCCC") if not sub.empty else "#CCCCCC"

dc_colors = [dominant_dp_color(dc) for dc in vis_dc]
sr_palette = ["#C0392B","#E91E63","#F1C40F","#03A9F4","#795548","#607D8B","#00897B","#FF7043"]
sr_colors  = sr_palette[:n_sr]

# ── Flow tables ───────────────────────────────────────────────────────────────
layer1 = df.groupby(["Design Principle", int_var]).size().reset_index(name="value")
layer2 = df.groupby([int_var, "Students reactions"]).size().reset_index(name="value")

# ── LEFT SANKEY: DC (right, x=0.99) → DP (left, x=0.01) ─────────────────────
left_nodes  = vis_dp + vis_dc
left_index  = {n: i for i, n in enumerate(left_nodes)}
left_labels = [""]*n_dp + [""]*n_dc   # all blanked — added as annotations
left_colors = dp_colors + dc_colors
left_x = [0.01]*n_dp + [0.99]*n_dc
left_y = col_y(n_dp) + dc_y

ls, lt, lv = [], [], []
for _, row in layer1.iterrows():
    dp, dc = row["Design Principle"], row[int_var]
    if dp not in left_index or dc not in left_index: continue
    ls.append(left_index[dc]); lt.append(left_index[dp]); lv.append(row["value"])

left_link_colors = [
    hex_to_rgba(left_colors[s])
    if (v > SHOW_MIN or matches(vis_dp[t] if t < n_dp else "", ALWAYS_SHOW)
        or matches(vis_dc[s - n_dp] if s >= n_dp else "", ALWAYS_SHOW))
    else "rgba(0,0,0,0)"
    for s, t, v in zip(ls, lt, lv)
]

# ── RIGHT SANKEY: DC (left, x=0.01) → SR (right, x=0.99) ────────────────────
all_sr_right = vis_sr + hide_sr
right_nodes  = vis_dc + all_sr_right
right_index  = {n: i for i, n in enumerate(right_nodes)}
right_labels = [""]*n_dc + [""]*n_sr + [""]*len(hide_sr)  # all blanked
right_colors = dc_colors + sr_colors + ["rgba(0,0,0,0)"]*len(hide_sr)
right_x = [0.01]*n_dc + [0.99]*n_sr + [0.99]*len(hide_sr)
right_y  = dc_y       + col_y(n_sr) + [0.01]*len(hide_sr)

rs, rt, rv, right_link_colors = [], [], [], []
for _, row in layer2.iterrows():
    dc, sr = row[int_var], row["Students reactions"]
    if dc not in right_index or sr not in right_index: continue
    rs.append(right_index[dc]); rt.append(right_index[sr]); rv.append(row["value"])
    is_ghost = sr in hide_sr
    visible  = (not is_ghost and (row["value"] > SHOW_MIN
                or matches(dc, ALWAYS_SHOW) or matches(sr, ALWAYS_SHOW)))
    right_link_colors.append(
        hex_to_rgba(right_colors[right_index[dc]]) if visible else "rgba(0,0,0,0)"
    )

# ── Figure ────────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Sankey(
    domain=dict(x=[0.04, 0.40], y=[0, 1]),
    arrangement="fixed",
    node=dict(pad=8, thickness=14,
              line=dict(color="white", width=0.5),
              label=left_labels, color=left_colors,
              x=left_x, y=left_y,
              hovertemplate="<b>%{label}</b><br>%{value}<extra></extra>"),
    link=dict(source=ls, target=lt, value=lv, color=left_link_colors,
              hovertemplate="<b>%{source.label}</b>→<b>%{target.label}</b><br>%{value}<extra></extra>"),
))

fig.add_trace(go.Sankey(
    domain=dict(x=[0.60, 0.96], y=[0, 1]),
    arrangement="fixed",
    node=dict(pad=8, thickness=14,
              line=dict(color="rgba(0,0,0,0)", width=0),
              label=right_labels, color=right_colors,
              x=right_x, y=right_y,
              hovertemplate="<b>%{label}</b><br>%{value}<extra></extra>"),
    link=dict(source=rs, target=rt, value=rv, color=right_link_colors,
              hovertemplate="<b>%{source.label}</b>→<b>%{target.label}</b><br>%{value}<extra></extra>"),
))

fig.update_layout(
    font=dict(family="Arial", size=13, color="#333333"),
    paper_bgcolor="white",
    height=600, width=1800,
    margin=dict(l=250, r=250, t=90, b=10),
)

# ── Descriptive Codes header (center top) ────────────────────────────────────
fig.add_annotation(x=0.5, y=1.06, xref="paper", yref="paper",
    text="<b>Descriptive Codes</b>", showarrow=False,
    font=dict(size=18, color="#444"), align="center")

# ── Outer left: Design Principles title ──────────────────────────────────────
fig.add_annotation(x=-0.17, y=0.5, xref="paper", yref="paper",
    text="<b>Design<br>Principles</b>", showarrow=False,
    font=dict(size=16, color="#333"), align="center",
    xanchor="center", yanchor="middle")

# ── Outer right: Student Reactions title ─────────────────────────────────────
fig.add_annotation(x=1.17, y=0.5, xref="paper", yref="paper",
    text="<b>Student<br>Reactions</b>", showarrow=False,
    font=dict(size=16, color="#333"), align="center",
    xanchor="center", yanchor="middle")

# Plotly Sankey node y uses y=0 at top (SVG convention); paper annotation y
# uses y=0 at bottom (math convention). Use 1-y_pos to align labels with bars.

# ── DP labels: left of left panel ────────────────────────────────────────────
for dp, y_pos in zip(vis_dp, col_y(n_dp)):
    fig.add_annotation(x=0.03, y=1 - y_pos, xref="paper", yref="paper",
        text=dp, showarrow=False,
        font=dict(size=11, color="#333333"),
        xanchor="right", yanchor="middle", align="right")

# ── DC labels: center gap ─────────────────────────────────────────────────────
for dc, y_pos in zip(vis_dc, dc_y):
    fig.add_annotation(x=0.5, y=1 - y_pos, xref="paper", yref="paper",
        text=wrap50(dc), showarrow=False,
        font=dict(size=9, color="#444444"),
        xanchor="center", yanchor="middle", align="center")

# ── SR labels: right of right panel ──────────────────────────────────────────
for sr, y_pos in zip(vis_sr, col_y(n_sr)):
    fig.add_annotation(x=0.97, y=1 - y_pos, xref="paper", yref="paper",
        text=sr, showarrow=False,
        font=dict(size=11, color="#333333"),
        xanchor="left", yanchor="middle", align="left")

# ── Arrows (centered over each panel) ────────────────────────────────────────
for x_pos, arrow in [(0.22, "<b>←</b>"), (0.78, "<b>→</b>")]:
    fig.add_annotation(x=x_pos, y=1.06, xref="paper", yref="paper",
        text=arrow, showarrow=False, font=dict(size=36, color="#2C7BB6"))

fig.write_html("sankey_twopanel.html")
print("✅  Saved: sankey_twopanel.html")
try:
    fig.write_image("sankey_twopanel.png", scale=2)
    print("✅  Saved: sankey_twopanel.png")
except Exception as e:
    print(f"⚠️  PNG export skipped ({e})")
fig.show()
